# Task 4 - Calibration, confidence tiers and directions

**Method.** The pin model reports `raw_spread` (its own sigma). For each group of addresses we collect the normalised score
`error / raw_spread` on data the model did not see and set `radius_p = Q(p) x raw_spread` (split conformal prediction with the
finite-sample correction). Three groups behave differently and are calibrated separately:

| group | meaning | calibration data | coverage is checked on |
|---|---|---|---|
| `visit` | address has its own cleaned visit evidence | surveyed out-of-fold errors, leave-one-out | the same rows, leave-one-out |
| `novisit` | locality known, no visit of its own | 1,231-address hide-and-predict set (visit pseudo-labels) | the 51 surveyed unvisited addresses |
| `noloc` | no locality in the text | hide-and-predict, unparsed subset | 6 surveyed addresses (too few to conclude) |

So for the groups that matter most, **the data used to set the radii and the data used to report coverage are different**.

In [1]:
import os
from pathlib import Path
if Path.cwd().name == 'notebooks':
    os.chdir(Path.cwd().parent)
import numpy as np, pandas as pd, matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from IPython.display import display
pd.set_option('display.width', 200); pd.set_option('display.max_columns', 30); pd.set_option('display.max_colwidth', 160)
R = Path('results')

## 1. Does a '90%' radius contain the truth about 90% of the time?

In [2]:
cov = pd.read_csv(R/'calibration_coverage.csv')
show = cov[cov.nominal.isin([0.5, 0.9])].copy()
display(show[['dimension','segment','n','nominal','coverage','ci95_low','ci95_high','median_radius_90','median_error']].round(3))

,dimension,segment,n,nominal,coverage,ci95_low,ci95_high,median_radius_90,median_error
0,overall,all surveyed,100,0.5,0.590,0.492,0.681,145.578,45.676
2,overall,all surveyed,100,0.9,0.910,0.838,0.952,145.578,45.676
3,group,noloc,6,0.5,0.667,0.300,0.903,1806.910,1082.123
5,group,noloc,6,0.9,0.667,0.300,0.903,1806.910,1082.123
6,group,novisit,51,0.5,0.647,0.510,0.764,347.459,110.999
8,group,novisit,51,0.9,0.941,0.841,0.980,347.459,110.999
9,group,visit,43,0.5,0.512,0.368,0.654,24.580,10.508
11,group,visit,43,0.9,0.907,0.784,0.963,24.580,10.508
12,town,T1,33,0.5,0.576,0.408,0.728,135.881,34.138
14,town,T1,33,0.9,0.848,0.691,0.933,135.881,34.138


In [3]:
rel = pd.read_csv(R/'calibration_reliability.csv')
fig, ax = plt.subplots(figsize=(5.2, 4.6))
ax.plot([0,1],[0,1],'k--',lw=1,label='perfect')
for name, g in rel.groupby('curve'):
    ax.plot(g.nominal, g.observed, marker='o', ms=4, label=f'{name} (n={int(g.n.iloc[0])})')
ax.set_xlabel('nominal coverage'); ax.set_ylabel('observed coverage'); ax.grid(alpha=.3); ax.legend(fontsize=7, loc='upper left')
ax.set_title('Reliability of the confidence radii'); plt.tight_layout(); plt.savefig(R/'calibration_reliability.png', dpi=140); plt.show()

C:\Users\ASUS\AppData\Local\Temp\ipykernel_26304\1167337464.py:7: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  ax.set_title('Reliability of the confidence radii'); plt.tight_layout(); plt.savefig(R/'calibration_reliability.png', dpi=140); plt.show()


Points above the diagonal are conservative (radius bigger than needed); below would be over-confident. Small groups wobble because of sample size, not necessarily miscalibration - the intervals above say how much.

## 2. Calibration by town and by the old geocoder's precision tier (90% radius)

In [4]:
sub = cov[(cov.nominal==0.9) & cov.dimension.isin(['town','old_geocoder_tier'])]
fig, ax = plt.subplots(figsize=(6.5, 3.2))
y = np.arange(len(sub))[::-1]
ax.errorbar(sub.coverage, y, xerr=[sub.coverage-sub.ci95_low, sub.ci95_high-sub.coverage], fmt='o', color='#3b6ea5', capsize=3)
ax.axvline(0.9, color='k', ls='--', lw=1); ax.set_yticks(y); ax.set_yticklabels([f'{d}: {s} (n={n})' for d,s,n in zip(sub.dimension, sub.segment, sub.n)], fontsize=8)
ax.set_xlabel('observed coverage of the 90% radius (95% Wilson interval)'); ax.set_xlim(0.2,1.02); ax.grid(alpha=.3)
plt.tight_layout(); plt.savefig(R/'calibration_by_segment.png', dpi=140); plt.show()
display(sub[['dimension','segment','n','coverage','ci95_low','ci95_high']].round(3))

C:\Users\ASUS\AppData\Local\Temp\ipykernel_26304\92613531.py:7: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.tight_layout(); plt.savefig(R/'calibration_by_segment.png', dpi=140); plt.show()


,dimension,segment,n,coverage,ci95_low,ci95_high
14,town,T1,33,0.848,0.691,0.933
17,town,T2,29,0.966,0.828,0.994
20,town,T3,38,0.921,0.792,0.973
23,old_geocoder_tier,locality,73,0.932,0.849,0.970
26,old_geocoder_tier,pincode,10,0.800,0.490,0.943
29,old_geocoder_tier,rooftop,1,1.000,0.207,1.000
32,old_geocoder_tier,street,16,0.875,0.640,0.965


Intervals for small strata (rooftop n=1, pincode n=10, the `noloc` group n=6) are wide and are **not** evidence either way.
Pincode-tier addresses with no locality in the text are the hard case; the model's radius for them is already in the kilometre range and their tier is always `low`.

## 3. Confidence tiers: do they separate easy from hard addresses?

In [5]:
tq = pd.read_csv(R/'confidence_tier_quality.csv'); display(tq.round(2))
pred = pd.read_csv('predictions.csv')
display(pred.groupby('confidence_tier').agg(addresses=('address_id','size'), median_radius_90=('radius_90','median'), median_location_confidence=('location_confidence','median')).round(2))
display(pred.groupby(['confidence_tier','suggested_action']).size().rename('addresses').to_frame())

,tier,n,median_err,within_100,within_250
0,high,39,9.57,0.97,1.0
1,low,35,192.45,0.20,0.6
2,medium,26,45.68,0.77,1.0


,addresses,median_radius_90,median_location_confidence
confidence_tier,,,
high,1106,21.20,0.98
low,1034,547.15,0.23
medium,740,157.25,0.77


,,addresses
confidence_tier,suggested_action,
high,VISIT_NOW,1106
low,VERIFY_FIRST,1034
medium,VISIT_WITH_DIRECTIONS,740


Rule: **high** = 90% radius <= 60 m (visit now); **medium** = <= 300 m (visit with directions); **low** = above that (verify first, never mark 'not traceable' on this basis).
On the surveyed addresses the tiers separate cleanly (accuracy columns above).

## 4. What drives the radius

In [6]:
fig, ax = plt.subplots(figsize=(6.5, 3.4))
order = pred.groupby('method_used').radius_90.median().sort_values().index
data = [np.log10(pred[pred.method_used==m].radius_90) for m in order]
ax.boxplot(data, vert=False, showfliers=False); ax.set_yticklabels([f'{m} (n={int((pred.method_used==m).sum())})' for m in order])
ax.set_xlabel('log10 of 90% radius (m)'); plt.tight_layout(); plt.savefig(R/'radius_by_method.png', dpi=140); plt.show()

C:\Users\ASUS\AppData\Local\Temp\ipykernel_26304\1962587679.py:4: MatplotlibDeprecationWarning: vert: bool was deprecated in Matplotlib 3.11 and will be removed in 3.13. Use orientation: {'vertical', 'horizontal'} instead.
  ax.boxplot(data, vert=False, showfliers=False); ax.set_yticklabels([f'{m} (n={int((pred.method_used==m).sum())})' for m in order])


C:\Users\ASUS\AppData\Local\Temp\ipykernel_26304\1962587679.py:5: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  ax.set_xlabel('log10 of 90% radius (m)'); plt.tight_layout(); plt.savefig(R/'radius_by_method.png', dpi=140); plt.show()


## 5. Directions

In [7]:
chk = pd.read_csv(R/'directions_checks.csv'); display(chk.round(3).T)
display(pred.sample(10, random_state=4)[['address_id','confidence_tier','radius_90','directions']])

,0
share_with_a_landmark,0.994
surveyed_with_address_landmark,72.000
address_landmark_within_300m_of_truth,0.625
address_landmark_within_300m_of_our_pin,0.639
same_landmark_chosen_for_pin_and_truth,0.972
share_directions_needing_locality_disambiguation,0.055


,address_id,confidence_tier,radius_90,directions
2648,AD002857,low,727.1,"APPROXIMATE (within ~730 m): 250 m SE of Masjid; also 350 m NW of Medical Store; also 400 m E of Ganesh Temple. Address: Block C, Road 2. Verify with a loca..."
2165,AD002342,low,605.9,APPROXIMATE (within ~610 m): 60 m SW of Govt School; also 300 m W of Water Tank; also 350 m NW of Park. Verify with a local before walking in.
2638,AD002847,medium,288.1,"Likely area (within ~290 m): 150 m S of Medical Store. Address: Block C, Road 1"
1109,AD001196,low,1932.4,"APPROXIMATE (within ~1930 m): 1150 m NW of Community Hall; also 450 m SW of Hanuman Temple; also 500 m S of Medical Store. Address: 9th Cross, 2nd Main. Ver..."
2727,AD002942,medium,213.9,Likely area (within ~210 m): 160 m N of Medical Store (Gandhi Basti side); also 400 m NE of Govt School. Address: Gali 9
2489,AD002691,high,50.7,"140 m NE of Ration Shop; also 400 m E of Medical Store; also 400 m E of Community Hall. Address: 10th Cross, 7th Main"
78,AD000084,low,325.8,"APPROXIMATE (within ~330 m): 150 m S of Masjid; also 300 m W of Ganesh Temple; also 450 m SE of Hanuman Temple. Address: Block B, Road 4. Verify with a loca..."
2252,AD002436,medium,248.4,"Likely area (within ~250 m): 200 m NE of Community Hall; also 350 m NW of Govt School; also 400 m SW of Petrol Bunk. Address: Gali 6, Ward 8"
2267,AD002453,high,19.1,"130 m NW of Church; also 190 m NE of Milk Dairy; also 250 m NE of Govt School. Address: Gali 8, Ward 11"
1993,AD002157,high,41.4,"80 m SE of Bus Stop; also 250 m N of Govt School; also 250 m N of Church. Address: Block D, Road 4"


Directions are plain-text templates built from the nearest landmarks to the pin (distance rounded, 8-point compass), with the address's own landmark first when one of that type is nearby, and the locality named when the same landmark name exists nearby. They need no network.
`test_task4.py` checks that 597 of 598 sampled direction clauses reproduce the distance and bearing from the stated landmark to the pin.
**Assumption to confirm with CN:** local x points east and y north (the source gives no CRS); flip `AXES` in `confidence_directions.py` if not.

## 6. Hand-off to Problem Statement 2

In [8]:
ps2 = pd.read_csv('ps2_location_confidence.csv'); display(ps2.head(6)); display(ps2.ps2_guidance.value_counts().to_frame())

,address_id,location_confidence,p_within_250m,radius_90_m,confidence_tier,hard_to_find_flag,reason_codes,ps2_guidance
0,AD000001,0.977,0.977,16.2,high,0,CONFIRMED_BY_FIELD_VISIT,LOCATION_CONFIRMED_OR_RELIABLE: a failed visit is informative
1,AD000002,0.977,0.977,24.1,high,0,CONFIRMED_BY_FIELD_VISIT,LOCATION_CONFIRMED_OR_RELIABLE: a failed visit is informative
2,AD000003,0.062,0.279,1224.4,low,1,INFERRED_FROM_STREET_NEIGHBOURS,LOCATION_UNRESOLVED: do not treat a failed visit here as evidence the address is invalid
3,AD000004,0.308,0.720,450.2,low,1,INFERRED_FROM_STREET_NEIGHBOURS,LOCATION_UNRESOLVED: do not treat a failed visit here as evidence the address is invalid
4,AD000005,0.202,0.605,579.4,low,1,INFERRED_FROM_LANDMARK,LOCATION_UNRESOLVED: do not treat a failed visit here as evidence the address is invalid
5,AD000007,0.373,0.780,392.0,low,1,INFERRED_FROM_STREET_NEIGHBOURS,LOCATION_UNRESOLVED: do not treat a failed visit here as evidence the address is invalid


,count
ps2_guidance,
LOCATION_CONFIRMED_OR_RELIABLE: a failed visit is informative,1106
LOCATION_UNRESOLVED: do not treat a failed visit here as evidence the address is invalid,1033
LOCATION_APPROXIMATE: a failed visit is weak evidence; retry with directions,741
